<!-- paper-paired-role-banner -->
> 🟦 **실습 코드 · 왼쪽 창** — TODO를 먼저 직접 구현하세요. 막히면 오른쪽 정답의 같은 셀에서 필요한 한 줄만 확인합니다.

# 03. ResNet — Deep Residual Learning for Image Recognition

- **원 논문:** [Deep Residual Learning for Image Recognition](https://arxiv.org/pdf/1512.03385)
- **저자 / 발표:** Kaiming He, Xiangyu Zhang, Shaoqing Ren, and Jian Sun · CVPR 2016 (2016)
- **난이도 / 예상 시간:** 중급 · 약 85분
- **선수 지식:** Conv2d, BatchNorm, autograd, matrix multiplication

이 노트북은 논문 결과 수치를 그대로 재현하는 대규모 benchmark가 아니라,
핵심 수식과 구조가 실제로 어떻게 동작하는지 확인하는 **하드웨어 인식 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하며, 작은 NumPy·Python 실험은
전송 비용을 피하기 위해 CPU에서 실행될 수 있습니다.

## 재현 범위와 완료 기준

**이 노트북이 재현하는 것**

Eq. (1)의 residual mapping H(x)=F(x)+x와 dimension-changing projection shortcut을 BasicBlock으로 구현한다. residual branch가 0일 때의 identity, stage shape, 깊이에 따른 직접 gradient path를 검증하고 작은 residual classifier를 합성 영상에 학습한다.

**원 논문과 다른 것**

원 논문은 ImageNet/CIFAR-10에서 18~152 layer networks와 224×224 images, 대규모 augmentation/training schedule을 비교한다. 여기서는 16×16 합성 영상, 8/16 channels, 3 basic blocks만 사용하며 benchmark 정확도나 degradation curve를 재현하지 않는다.

완료하려면 TODO를 구현하고, 검증 셀의 `assert`를 모두 통과시키고, 마지막 관찰 질문에
자신의 말로 답하세요. 수치가 논문과 다르더라도 핵심 불변식과 비교 방향이 맞으면 성공입니다.

## 논문 ↔ 노트북 지도

        절 번호와 수식 번호는 위 링크의 대표 공개본을 기준으로 합니다. 판본에 따라 페이지는 달라질 수
        있으므로 **절 제목과 수식 번호**를 함께 사용하세요.

        | 원 논문의 위치 | 이 노트북의 대응 실습 | 확인할 증거 |
        |---|---|---|
        | §3.1, Eq. (1): H(x)=F(x)+x | `BasicBlock.forward`의 `out + self.shortcut(x)` | residual branch를 0으로 만들었을 때 양수 입력 identity를 assert |
| §3.2, Eq. (2): y=F(x,{Wi})+Ws x projection shortcut | stride/channel 변화 때의 1×1 Conv+BN shortcut | 8→16 channels, stride 2 block의 `(N,16,8,8)` shape 검증 |
| §3.3 및 Fig. 3: 3×3 두 개로 이루어진 residual building block | `conv1-bn-relu-conv2-bn-add-relu` | block 모듈과 `MiniResNet` stage 구성을 코드로 확인 |
| §4.1 ‘Plain Networks’/‘Residual Networks’: shortcut의 optimization path | 동일한 작은 linear transform을 깊게 합성한 gradient 실험 | plain 0.1^L 경로와 residual (I+0.1I)^L 경로의 input-gradient plot |

        > 읽기 순서: 먼저 해당 절을 훑고 → 코드를 예측해 작성하고 → 출력이 논문의 주장과 왜
        > 연결되는지 한 문장으로 설명합니다.

## 핵심 수식과 구현 설계

$$
y=\mathcal{F}(x;W)+x
$$

- $\mathcal{F}$는 residual branch가 학습하는 변환이고 $x$는 identity shortcut입니다.
- 목표는 전체 mapping을 새로 학습하기보다 입력에 더할 잔차를 학습하는 것입니다.
- residual block class가 main branch와 shortcut projection을 명시적으로 구분합니다.
- 더셈을 위해 두 branch의 shape이 같아야 하며, stride/channel 변화 시 projection을 사용합니다.

## 구현 선택 이유

- **원문에서 보존한 부분:** §3.1, Eq. (1): H(x)=F(x)+x
- **노트북 구현:** `BasicBlock.forward`의 `out + self.shortcut(x)`
- **이렇게 구현한 이유:** 완성된 고수준 model을 한 줄로 호출하면
  논문의 핵심 shape·mask·loss·상태 변화가 숨겨집니다. 따라서 이 계산을
  이름이 드러나는 class와 함수로 나누고 중간 tensor를 assertion으로
  검사합니다.
- **실패를 잡는 증거:** residual branch를 0으로 만들었을 때 양수 입력 identity를 assert를 출력 크기만 아니라
  구현 불변식으로 사용합니다.
- **축소 선택:** 원 논문은 ImageNet/CIFAR-10에서 18~152 layer networks와 224×224 images, 대규모 augmentation/training schedule을 비교한다. 여기서는 16×16 합성 영상, 8/16 channels, 3 basic blocks만 사용하며 benchmark 정확도나 degradation curve를 재현하지 않는다.는 기본 실행에서 생략합니다.
  이는 Windows CPU에서도 반복 실험이 가능하게 하면서 핵심 메커니즘은
  유지하기 위한 선택입니다.

## 1. Residual learning의 최소 단위

네트워크가 목표 함수 `H(x)` 전체를 직접 맞추는 대신 `F(x)=H(x)-x`를 학습하고
출력에서 `x`를 더합니다. shortcut에는 parameter가 없고 shape가 달라질 때만 Eq. (2)의
projection `W_s`를 씁니다. 원 논문의 post-activation basic block을 구현하므로 덧셈 뒤
ReLU가 있습니다. 따라서 ‘정확한 identity’ 검증에는 비음수가 아닌 입력을 사용합니다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** input/shortcut [B,C,H,W] → residual output [B,C',H',W'] → logits [B,K]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(3)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device


def make_quadrant_images(repeats=8, size=16):
    generator = torch.Generator().manual_seed(404)
    images, labels = [], []
    locations = ((2, 2), (2, 10), (10, 2), (10, 10))
    for index in range(4 * repeats):
        label = index % 4
        image = torch.zeros(3, size, size)
        row, col = locations[label]
        image[label % 3, row : row + 4, col : col + 4] = 1.0
        noise = 0.04 * torch.randn(image.shape, generator=generator)
        images.append((image + noise).clamp(0, 1))
        labels.append(label)
    return torch.stack(images), torch.tensor(labels)


res_x, res_y = make_quadrant_images()
res_x, res_y = ACCELERATOR.move(res_x, res_y)
assert res_x.shape == (32, 3, 16, 16)
print(ACCELERATOR.summary())
print("residual toy batch:", tuple(res_x.shape))

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §3.1, Eq. (1): H(x)=F(x)+x / §3.2, Eq. (2): y=F(x,{Wi})+Ws x projection shortcut / §3.3 및 Fig. 3: 3×3 두 개로 이루어진 residual building block
- **이 셀의 책임:** `BasicBlock.forward`의 `out + self.shortcut(x)` / stride/channel 변화 때의 1×1 Conv+BN shortcut / `conv1-bn-relu-conv2-bn-add-relu`
- **Tensor shape 계약:** input/shortcut [B,C,H,W] → residual output [B,C',H',W'] → logits [B,K]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** residual branch를 0으로 만들었을 때 양수 입력 identity를 assert / 8→16 channels, stride 2 block의 `(N,16,8,8)` shape 검증 / block 모듈과 `MiniResNet` stage 구성을 코드로 확인. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO: 두 3×3 Conv-BN과 shortcut을 갖는 post-activation BasicBlock을 구현하세요.
class BasicBlock(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self, in_channels, out_channels, stride=1):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def forward(self, x):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 3 · 논문 주장 확인 실험

- **원문 위치:** §3.1, Eq. (1): H(x)=F(x)+x / §3.2, Eq. (2): y=F(x,{Wi})+Ws x projection shortcut
- **이 셀의 책임:** `BasicBlock.forward`의 `out + self.shortcut(x)` / stride/channel 변화 때의 1×1 Conv+BN shortcut
- **Tensor shape 계약:** input/shortcut [B,C,H,W] → residual output [B,C',H',W'] → logits [B,K]
- **구현 이유:** 한 변수만 바꾸어 해당 메커니즘이 출력에 미치는 영향을 분리합니다.
- **완료 증거:** residual branch를 0으로 만들었을 때 양수 입력 identity를 assert / 8→16 channels, stride 2 block의 `(N,16,8,8)` shape 검증. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO: identity block의 residual branch를 0으로 만들고 양수 입력이 보존되는지 확인하세요.
# 이어서 8→16, stride=2 projection block의 shape를 assert하세요.
_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."

### 구현 단계 4 · 핵심 연산·모델

- **원문 위치:** §3.3 및 Fig. 3: 3×3 두 개로 이루어진 residual building block
- **이 셀의 책임:** `conv1-bn-relu-conv2-bn-add-relu`
- **Tensor shape 계약:** input/shortcut [B,C,H,W] → residual output [B,C',H',W'] → logits [B,K]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** block 모듈과 `MiniResNet` stage 구성을 코드로 확인. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO: stem(3→8), identity blocks, 8→16 projection, global average pool, FC로
class MiniResNet(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self, classes=4):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def forward(self, x):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


resnet = MiniResNet().to(DEVICE)


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 5 · 평가·완료 증거

- **원문 위치:** §4.1 ‘Plain Networks’/‘Residual Networks’: shortcut의 optimization path
- **이 셀의 책임:** 동일한 작은 linear transform을 깊게 합성한 gradient 실험
- **Tensor shape 계약:** input/shortcut [B,C,H,W] → residual output [B,C',H',W'] → logits [B,K]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** plain 0.1^L 경로와 residual (I+0.1I)^L 경로의 input-gradient plot. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO: depth 0..12에서 h←0.1Ih(plain)와 h←h+0.1Ih(residual)의
def input_gradient_norm(depth, residual):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 6 · 평가·완료 증거

- **원문 위치:** §3.1, Eq. (1): H(x)=F(x)+x / §4.1 ‘Plain Networks’/‘Residual Networks’: shortcut의 optimization path
- **이 셀의 책임:** `BasicBlock.forward`의 `out + self.shortcut(x)` / 동일한 작은 linear transform을 깊게 합성한 gradient 실험
- **Tensor shape 계약:** input/shortcut [B,C,H,W] → residual output [B,C',H',W'] → logits [B,K]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** residual branch를 0으로 만들었을 때 양수 입력 identity를 assert / plain 0.1^L 경로와 residual (I+0.1I)^L 경로의 input-gradient plot. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO: 합성 quadrant batch에 30 step 학습하고 loss/accuracy를 보고하세요.
_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."

## 마무리 관찰 기록

아래 세 문장을 직접 완성하세요.

1. 이 논문의 핵심 연산은 `_____`이고, 코드에서는 `_____`에 해당한다.
2. 원 규모 실험 대신 미니 재현을 사용했기 때문에 확인할 수 없는 주장은 `_____`이다.
3. 한 가지 변수를 바꾸어 다시 실행한다면 `_____`를 바꾸고 `_____`를 측정하겠다.

**추가 실험:** seed는 유지한 채 한 변수만 바꾸고, 변경 전후의 metric을 표로 남기세요.